# Pgvecto_rs embedding storage
In this notebook i show how you can connect pgvecto to llama_index and be able to query the db to get the embeddings stored

In [1]:
# i will set up the logger so the processing will be logged in the console to better understand what is happening and be able to debug properly

import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [7]:
# we initialize the env var from the parent directory of the current
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [8]:
from pgvecto_rs.sdk import PGVectoRs
from llama_index.vector_stores.pgvecto_rs import PGVectoRsStore

DATABASE_URL = "postgresql+psycopg://{username}:{password}@{host}:{port}/{db_name}".format(
    port=os.getenv("DB_PORT", "5432"),
    host=os.getenv("DB_HOST", "localhost"),
    username=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASS", "mysecretpassword"),
    db_name=os.getenv("DB_NAME", "postgres"),
)

client = PGVectoRs(
    db_url=DATABASE_URL,
    collection_name="items",
    dimension=1536,  # Using OpenAI’s text-embedding-ada-002
)

vector_store = PGVectoRsStore(client=client)


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\pgvecto_rs\sdk\client.py:36: SAWarning: This declarative base already contains a class with the same class name and module name as pgvecto_rs.sdk.client._Table, and will be replaced in the string-lookup table.
  class _Table(RecordORM):


In [9]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline, DocstoreStrategy
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.storage.docstore.postgres import PostgresDocumentStore

documents = SimpleDirectoryReader(input_files=["../data/paul_graham_essay.txt"]).load_data()

# the splitter that will be used to create chunks from the doc
splitter = SentenceSplitter(
    chunk_size=520,
    chunk_overlap=50
)

# the model that will be used to make the embeddings from the nodes
embedder = OpenAIEmbedding(model="text-embedding-ada-002")

pipeline = IngestionPipeline(
    transformations=[
        splitter,
        embedder,
    ],
    vector_store=vector_store,
    # i use the docstore to track files that have already been processed
    docstore=PostgresDocumentStore.from_uri(uri=DATABASE_URL, table_name="items_docstore"),
    docstore_strategy=DocstoreStrategy.UPSERTS # so that we update if the hash is different else we skip
)

pipeline.run(documents=documents)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[TextNode(id_='c28007d4-bd26-4307-8049-85566c92748c', embedding=[-0.006544576026499271, -0.010054666548967361, -0.009297719225287437, -0.035368844866752625, -0.0024215602315962315, 0.029661599546670914, -0.027022333815693855, -0.017818395048379898, -0.019908372312784195, -0.019359083846211433, 0.026285482570528984, 0.035368844866752625, -0.01816672459244728, -0.010758023709058762, 0.001813658163882792, 0.016612639650702477, 0.022507447749376297, 0.013095851056277752, 0.007013481110334396, -0.016076747328042984, -0.0112001346424222, 0.00032237230334430933, 0.012178136967122555, 0.008982882834970951, -0.008982882834970951, -0.007448893040418625, 0.013129344210028648, -0.033118102699518204, 0.005894807633012533, -0.008647950366139412, 0.013665235601365566, -0.01602315902709961, 0.0029674996621906757, -0.014294908381998539, -0.03271618112921715, -0.010349406860768795, -0.012667138129472733, 0.0021720356307923794, 0.004498139955103397, -0.014428880997002125, 0.037137288600206375, 0.01619732

In [10]:
from llama_index.core import VectorStoreIndex

# let us now try to quickly query the table

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

query_engine = index.as_query_engine()

response = query_engine.query("What would you say is the most important\
                               thing that the writer would want me to know?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [11]:
print(response)

The writer would want you to know about their experiences and insights gained from working on various projects, including writing essays, programming, hosting dinners, and engaging in startup-related activities.


## Observations
### Database tables
Two database tables have been created from this:
- collection_items
- data_items_docstore

### Database indexes
- collection_items_pkey - on the `collection_items` table
- data_items_docstore:idx_key_namespace - on the `data_items_docstore` table
- data_items_docstore:unique_key_namespace - on the `data_items_docstore` table
- data_items_docstore_pkey - on the `data_items_docstore` table

### collection_items columns
- id - primary_key
- text - the chunk text stored
- meta - meta data based on the document
- embedding - the embedded text stored as vectors with scalar 1536 which aligns with the embedding model we used of text-embedding-ada-002

### data_items_docstore columns
- id
- key
- namespace
- value - JSON data

The tablenames i specified have slightly been altered,
`items` -> `collection_items`
`items_docstore` -> `data_items_docstore`
This leads to my thought that when specifying the table names you can actually use the same table name and then the names will be altered and will be different in the db when stored.
Say i used the name items for both the storage for the embeddings and the docstore i would get
`collection_items` and `data_items` in the db but usage would still remain as me using items in code when using llama_index. cool stuff

### Why use a database
The reason why using a database is important is that it allows you to create the embeddings once and simply use them when the agent is queried, this will lead to:
1. Faster query processing
2. reduced cost as the embedding do not have to be created every time using a paid LLM as openai
